In [1]:
import os
import pandas as pd
import glob
import matplotlib.pyplot as plt
import duckdb
import polars as pl
import altair as alt

In [2]:
path = 'sample_data/service_level'

# Get a list of all CSV files in the folder
all_files = glob.glob(os.path.join(path, "*.csv"))

# Create an empty list to store individual DataFrames
df_list = []

# Loop through the file paths, read each file, and append the DataFrame to the list
for filename in all_files:
    df = pd.read_csv(filename, index_col=None, header=0)
    df_list.append(df)

# Concatenate all DataFrames in the list into a single DataFrame
merged_df = pd.concat(df_list, ignore_index=True)

# Convert column names to lowercase and snake case
merged_df.columns = merged_df.columns.str.lower().str.replace(' ', '_')
merged_df.to_csv('sample_data/combined_service_level_data.csv', index=False)

# Display the combined DataFrame
combined_df = pd.read_csv('sample_data/combined_service_level_data.csv', index_col=None, header=0)
combined_df.head()

,iso3,country,residence_type,service_type,year,coverage,population,service_level
0,ABW,Aruba,total,Sanitation,2019,98.72179,105707.34211,At least basic
1,ABW,Aruba,total,Hygiene,2019,98.60724,105584.69160,Basic service
2,ABW,Aruba,total,Hygiene,2019,1.22276,1309.27920,Limited service
3,ABW,Aruba,total,Sanitation,2019,0.00000,0.00000,Limited service
4,ABW,Aruba,total,Hygiene,2019,0.17000,182.02920,No handwashing facility


In [3]:
# Select specific countries and generate list from set object
service_type_unique = set(combined_df['service_type'])
service_type_to_plot = list(service_type_unique)
service_type_to_plot

['Hygiene', 'Sanitation', 'Drinking water']

In [4]:
combined_df['coverage'].max()

100.0

In [5]:
# Convert to Polars
df_polars = pl.from_pandas(combined_df)

In [6]:
df_agg_service_type_polars = df_polars.group_by(["year", "service_type"]).agg(
                  pl.col("service_type").count().alias("service_type_count"),
              ).sort(['year'], descending=[False])

print(df_agg_service_type_polars)

shape: (75, 3)
┌──────┬────────────────┬────────────────────┐
│ year ┆ service_type   ┆ service_type_count │
│ ---  ┆ ---            ┆ ---                │
│ i64  ┆ str            ┆ u32                │
╞══════╪════════════════╪════════════════════╡
│ 2000 ┆ Sanitation     ┆ 2572               │
│ 2000 ┆ Hygiene        ┆ 129                │
│ 2000 ┆ Drinking water ┆ 2586               │
│ 2001 ┆ Sanitation     ┆ 2613               │
│ 2001 ┆ Drinking water ┆ 2604               │
│ …    ┆ …              ┆ …                  │
│ 2023 ┆ Sanitation     ┆ 2581               │
│ 2023 ┆ Drinking water ┆ 2623               │
│ 2024 ┆ Hygiene        ┆ 770                │
│ 2024 ┆ Drinking water ┆ 2604               │
│ 2024 ┆ Sanitation     ┆ 2537               │
└──────┴────────────────┴────────────────────┘


In [7]:
df_agg_service_level_polars = df_polars.group_by(["year", "service_level"]).agg(
                  pl.col("service_level").count().alias("service_level_count"),
              ).sort(['year'], descending=[False])

print(df_agg_service_level_polars)

shape: (200, 3)
┌──────┬─────────────────────────┬─────────────────────┐
│ year ┆ service_level           ┆ service_level_count │
│ ---  ┆ ---                     ┆ ---                 │
│ i64  ┆ str                     ┆ u32                 │
╞══════╪═════════════════════════╪═════════════════════╡
│ 2000 ┆ No handwashing facility ┆ 60                  │
│ 2000 ┆ Limited service         ┆ 1154                │
│ 2000 ┆ Surface water           ┆ 554                 │
│ 2000 ┆ Safely managed service  ┆ 690                 │
│ 2000 ┆ At least basic          ┆ 432                 │
│ …    ┆ …                       ┆ …                   │
│ 2024 ┆ Limited service         ┆ 1357                │
│ 2024 ┆ Unimproved              ┆ 1115                │
│ 2024 ┆ No handwashing facility ┆ 265                 │
│ 2024 ┆ Open defecation         ┆ 532                 │
│ 2024 ┆ Surface water           ┆ 554                 │
└──────┴─────────────────────────┴─────────────────────┘


In [8]:
# Convert to a pandas DataFrame
df_agg_result_service_type = df_agg_service_type_polars.to_pandas()
df_agg_result_service_level = df_agg_service_level_polars.to_pandas()
#df_agg_result_service_type.head()
#df_agg_result_service_level.head()

In [9]:
alt.Chart(df_agg_result_service_type).mark_bar(size=20).encode(
    x=alt.X('year', axis=alt.Axis(title='Year', format='d')),
    y='service_type_count',
    #color='service_type',
    color=alt.Color('service_type', legend=alt.Legend(title='Service type')),
).properties(
    width=1000,
    title='Service Type breakdown Totals Over Time (2000-2024)'
)

alt.Chart(...)

In [10]:
alt.Chart(df_agg_result_service_level).mark_bar(size=20).encode(
    x=alt.X('year', axis=alt.Axis(title='Year', format='d')),
    y='service_level_count',
    color='service_level',

).properties(
    width=1000,
    title='Service Level breakdown Totals Over Time (2000-2024)'
)

alt.Chart(...)

In [11]:
alt.Chart(df_agg_result_service_level).mark_line(size=2).encode(
    #x='year',
    x=alt.X('year', axis=alt.Axis(title='Year', format='d')),
    y='service_level_count',
    color='service_level',

).properties(
    width=1000,
    title='Service Level breakdown Totals Over Time (2000-2024)'
)

alt.Chart(...)